# 49 — E1 method heatmap, unquantized

Regenerates the E1 "mean joint-z by method and rho stratum" heatmap from the
unquantized sweep results, as a companion to the quantized figure produced by
`22_results_report.ipynb`.

- Input: `resources/20_e1_family_sweep_unquantized.jsonl` (540 records)
- Pairs: `resources/15_e0_selected_pairs.json` (shared by both pipelines)
- Output: `resources/22_e1_method_heatmap_unquantized.png`

The `frame` / `add_joint_z` / `by_stratum` helpers are copied verbatim from
`22_results_report.ipynb` so the joint-z numbers are computed identically.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

SELECTION = json.loads(Path("resources/15_e0_selected_pairs.json").read_text())[
    "selected"
]

records = [
    json.loads(line)
    for line in Path("resources/20_e1_family_sweep_unquantized.jsonl")
    .read_text()
    .strip()
    .split("\n")
]
assert len(records) == 540, f"expected 540 records, got {len(records)}"
print(f"loaded {len(records)} records")

In [ ]:
def frame(records: list[dict], a_of: dict, b_of: dict, keys: tuple) -> pd.DataFrame:
    """Flatten harness records; a/b synsets resolved per record via callables."""
    rows = []
    for i, r in enumerate(records):
        c = r["config"]
        a, b = a_of(c), b_of(c)
        rows.append(
            {
                **{key: c.get(key) for key in keys},
                "prompt": r["prompt"],
                "a": a,
                "b": b,
                "delta_a": r["expression"][a]["delta"],
                "delta_b": r["expression"][b]["delta"],
                "ppl_ratio": r["ppl_ratio"],
                "fluency_flag": r["fluency_flag"],
                "text": r["text"],
                "row": i,
            }
        )
    return pd.DataFrame(rows)


def add_joint_z(df: pd.DataFrame) -> pd.DataFrame:
    """z per concept over the whole experiment; joint = min (AND semantics)."""
    for side in ("a", "b"):
        z = df.groupby(side)[f"delta_{side}"].transform(
            lambda g: (g - g.mean()) / (g.std() + 1e-8)
        )
        df[f"z_{side}"] = z
    df["joint_z"] = df[["z_a", "z_b"]].min(axis=1)
    return df


def by_stratum(side: str):
    return lambda c: SELECTION[c["stratum"]][side]

In [ ]:
e1 = add_joint_z(
    frame(
        records,
        by_stratum("a"),
        by_stratum("b"),
        ("stratum", "rho", "method", "family"),
    )
)
e1_summary = (
    e1.groupby(["stratum", "rho", "method", "family"])
    .agg(joint_z=("joint_z", "mean"))
    .reset_index()
)

heat = e1_summary.pivot_table(
    index="method", columns="stratum", values="joint_z"
).reindex(columns=["low", "medium", "high"])
fig, ax = plt.subplots(figsize=(6.5, 4.5))
im = ax.imshow(heat.values, cmap="RdYlGn", aspect="auto")
ax.set_xticks(range(3), heat.columns)
ax.set_yticks(range(len(heat)), heat.index)
for i in range(len(heat)):
    for j in range(3):
        ax.text(
            j, i, f"{heat.values[i, j]:.2f}", ha="center", va="center", fontsize=8
        )
ax.set_title("E1: mean joint-z by method and rho stratum (unquantized)")
fig.colorbar(im, shrink=0.8)
plt.tight_layout()
plt.savefig("resources/22_e1_method_heatmap_unquantized.png", dpi=150)
plt.show()

display(heat.round(2))